# QFT-Graph: Training on Google Colab

End-to-end training pipeline for the heterogeneous GNN on scalar φ⁴ theory.

**Steps:**
1. Mount Google Drive & install dependencies
2. Generate Monte Carlo training data
3. Build heterogeneous graph dataset
4. Train the HeteroGNN model
5. Evaluate energy prediction quality
6. Run coupling sweep for critical exponents

## 0. Setup: Mount Google Drive & Install Dependencies

In [ ]:
import os
import sys

IN_COLAB = 'COLAB_GPU' in os.environ or 'COLAB_RELEASE_TAG' in os.environ

if IN_COLAB:
    # Mount Google Drive
    from google.colab import drive
    drive.mount('/content/drive')

    # Path to the project on your Google Drive
    PROJECT_ROOT = '/content/drive/MyDrive/qft_graph'

    # Install dependencies (Colab already has PyTorch)
    !pip install -q torch-geometric omegaconf

    # Add project source to Python path
    sys.path.insert(0, os.path.join(PROJECT_ROOT, 'src'))

    # Set working directory so relative paths (data/, experiments/) resolve inside the project
    os.chdir(PROJECT_ROOT)
    print(f'Working directory: {os.getcwd()}')
else:
    # Local: just add src to path
    sys.path.insert(0, '../src')

print('Setup complete.')

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from qft_graph.config import (
    LatticeConfig, ScalarFieldConfig, MCConfig, ModelConfig, TrainingConfig
)
from qft_graph.lattice.hypercubic import HypercubicLattice
from qft_graph.fields.scalar import ScalarField
from qft_graph.actions.phi4 import Phi4Action
from qft_graph.mc.metropolis import MetropolisSampler
from qft_graph.mc.observables import ObservableSet
from qft_graph.mc.analysis import jackknife_mean_error
from qft_graph.graphs.builder import HeteroGraphBuilder
from qft_graph.models.hetero_gnn import HeteroGNN
from qft_graph.training.trainer import Trainer
from qft_graph.training.metrics import energy_correlation, relative_error
from qft_graph.utils.reproducibility import set_seed

set_seed(42)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name()}')

plt.style.use('dark_background')
%matplotlib inline

## 1. Configuration

All parameters in one place for easy experimentation.

In [ ]:
# === LATTICE ===
LATTICE_SIZE = (16, 16)       # 2D square lattice
LATTICE_SPACING = 1.0

# === PHYSICS ===
MASS_SQUARED = -0.5           # Bare mass squared (near critical)
COUPLING = 0.5                # Quartic coupling lambda

# === MONTE CARLO ===
N_CONFIGS = 5000              # Training configurations
N_THERMALIZATION = 1000       # Thermalization sweeps
N_SWEEPS_BETWEEN = 10         # Decorrelation sweeps
MC_STEP_SIZE = 1.0            # Metropolis proposal width

# === MODEL ===
HIDDEN_DIM = 64               # GNN hidden dimension
N_MP_BLOCKS = 3               # Number of 3-stage message passing blocks
ENCODER_LAYERS = 2            # MLP depth in encoders

# === TRAINING ===
EPOCHS = 150
BATCH_SIZE = 32
LEARNING_RATE = 1e-3
LOSS_FN = 'energy_matching'   # 'energy_matching' | 'kl_divergence' | 'relative_energy'

print(f'Lattice: {LATTICE_SIZE[0]}x{LATTICE_SIZE[1]}, a={LATTICE_SPACING}')
print(f'Physics: m²={MASS_SQUARED}, λ={COUPLING}')
print(f'MC: {N_CONFIGS} configs, {N_THERMALIZATION} therm, {N_SWEEPS_BETWEEN} decorr')
print(f'Model: H={HIDDEN_DIM}, {N_MP_BLOCKS} MP blocks')
print(f'Training: {EPOCHS} epochs, batch={BATCH_SIZE}, lr={LEARNING_RATE}')

## 2. Load or Generate Monte Carlo Data

Pre-generated data loads instantly from `data/mc_configs/`. If not found, generates fresh data (~5 min for 5000 configs on 16x16).

**To pre-generate offline**, run from the project root:
```bash
python scripts/generate_mc_data.py --dimensions 16 16 --mass_squared -0.5 --coupling 0.5 --n_configs 5000
```

In [ ]:
lattice_config = LatticeConfig(dimensions=LATTICE_SIZE, spacing=LATTICE_SPACING)
lattice = HypercubicLattice(lattice_config)
field_config = ScalarFieldConfig(mass_squared=MASS_SQUARED, coupling=COUPLING)
action = Phi4Action(lattice, field_config)

# Try loading pre-generated data first
dims_str = "x".join(str(d) for d in LATTICE_SIZE)
data_path = Path(f"data/mc_configs/phi4_{dims_str}_m2={MASS_SQUARED}_lam={COUPLING}/mc_data.pt")

if data_path.exists():
    print(f"Loading pre-generated data from {data_path}")
    mc_data = torch.load(data_path, weights_only=False)
    configurations = mc_data["configurations"][:N_CONFIGS]
    actions = mc_data["actions"][:N_CONFIGS]
    acceptance_rate = mc_data.get("acceptance_rate", 0.0)
    print(f"Loaded {len(configurations)} configurations (acceptance rate: {acceptance_rate:.3f})")
else:
    print(f"No pre-generated data at {data_path}")
    print(f"Generating {N_CONFIGS} configs on {LATTICE_SIZE} lattice (this may take a few minutes)...")
    mc_config = MCConfig(
        n_configs=N_CONFIGS,
        n_thermalization=N_THERMALIZATION,
        n_sweeps_between=N_SWEEPS_BETWEEN,
        step_size=MC_STEP_SIZE,
        seed=42,
    )
    sampler = MetropolisSampler(action, mc_config)
    mc_result = sampler.generate(N_CONFIGS)
    configurations = mc_result.configurations
    actions = mc_result.actions
    acceptance_rate = mc_result.acceptance_rate

    # Save for next time
    data_path.parent.mkdir(parents=True, exist_ok=True)
    torch.save({
        "configurations": configurations,
        "actions": actions,
        "acceptance_rate": acceptance_rate,
    }, data_path)
    print(f"Saved to {data_path} for future runs")

print(f"\nConfigurations: {configurations.shape}")
print(f"Actions - mean: {actions.mean():.2f}, std: {actions.std():.2f}")

In [ ]:
# Quick sanity checks
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Action distribution
axes[0].hist(actions.numpy(), bins=50, alpha=0.7, color='#4488ff')
axes[0].set_xlabel(r'$S_E[\phi]$')
axes[0].set_ylabel('Count')
axes[0].set_title('Action Distribution')

# Field value distribution
axes[1].hist(configurations.flatten().numpy(), bins=100, alpha=0.7, color='#44bb88', density=True)
axes[1].set_xlabel(r'$\phi$')
axes[1].set_ylabel('Density')
axes[1].set_title('Field Value Distribution')

# Sample configuration heatmap
axes[2].imshow(configurations[0].reshape(LATTICE_SIZE).numpy(),
               cmap='RdBu_r', origin='lower')
axes[2].set_title('Sample Configuration')
axes[2].set_xlabel('x')
axes[2].set_ylabel('y')

plt.tight_layout()
plt.show()

## 3. Build Heterogeneous Graph Dataset

Convert MC configurations into PyG HeteroData objects with the bipartite spacetime/field structure.

In [ ]:
scalar_field = ScalarField()
builder = HeteroGraphBuilder(lattice, [scalar_field])

print('Building graph dataset...')
dataset = builder.build_dataset(
    configurations={'scalar': configurations},
    actions=actions,
)
print(f'Dataset size: {len(dataset)} graphs')

# Inspect one graph
sample = dataset[0]
print(f'\nSample graph:')
print(f'  Node types: {sample.node_types}')
print(f'  Edge types: {sample.edge_types}')
print(f'  Spacetime features: {sample["spacetime"].x.shape}')
print(f'  Scalar features: {sample["scalar"].x.shape}')
print(f'  Target action: {sample.y.item():.4f}')

In [ ]:
# Train/val split
n_train = int(0.8 * len(dataset))
train_dataset = dataset[:n_train]
val_dataset = dataset[n_train:]
print(f'Train: {len(train_dataset)}, Val: {len(val_dataset)}')

## 4. Create and Train the HeteroGNN

In [ ]:
model_config = ModelConfig(
    hidden_dim=HIDDEN_DIM,
    n_mp_blocks=N_MP_BLOCKS,
    encoder_layers=ENCODER_LAYERS,
    dropout=0.0,
    activation='gelu',
    readout='energy',
)

model = HeteroGNN(
    config=model_config,
    lattice_dim=lattice.dimension(),
    field_types={'scalar': scalar_field.dof_per_site()},
    lattice_spacing=lattice.lattice_spacing(),
)

n_params = sum(p.numel() for p in model.parameters())
print(f'Model parameters: {n_params:,}')
print(f'\nArchitecture:')
print(f'  Encoders: spacetime ({lattice.dimension()+1} -> {HIDDEN_DIM}), scalar (1 -> {HIDDEN_DIM}), edge ({lattice.dimension()} -> {HIDDEN_DIM})')
print(f'  Message Passing: {N_MP_BLOCKS} x ThreeStageBlock (field->ST->ST->field)')
print(f'  Readout: EnergyHead (per-site MLP summed over lattice)')

In [ ]:
training_config = TrainingConfig(
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    lr=LEARNING_RATE,
    weight_decay=1e-5,
    scheduler='cosine',
    loss=LOSS_FN,
    checkpoint_every=50,
    seed=42,
)

experiment_dir = Path('experiments/runs/colab_run')

trainer = Trainer(
    model=model,
    train_dataset=train_dataset,
    val_dataset=val_dataset,
    config=training_config,
    experiment_dir=experiment_dir,
    device=device,
)

print(f'Training on {device} for {EPOCHS} epochs...')
history = trainer.train()

print(f'\nFinal train loss: {history["train_loss"][-1]:.6f}')
print(f'Final val loss: {history["val_loss"][-1]:.6f}')
print(f'Final val correlation: {history["val_corr"][-1]:.4f}')

In [ ]:
# Plot training curves
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

epochs = range(1, len(history['train_loss']) + 1)

axes[0].semilogy(epochs, history['train_loss'], label='Train', alpha=0.8)
axes[0].semilogy(epochs, history['val_loss'], label='Val', alpha=0.8)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(epochs, history['val_corr'], color='#44bb88')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Pearson Correlation')
axes[1].set_title('Energy Prediction Correlation')
axes[1].set_ylim([0, 1.05])
axes[1].grid(True, alpha=0.3)

plt.suptitle(f'Training: {LATTICE_SIZE[0]}x{LATTICE_SIZE[1]}, m²={MASS_SQUARED}, λ={COUPLING}', fontsize=13)
plt.tight_layout()
plt.show()

## 5. Evaluate Energy Predictions

In [ ]:
from torch_geometric.loader import DataLoader as PyGDataLoader

model.eval()
all_pred = []
all_true = []

# Rebuild val dataset to get clean (un-encoded) features
# (No longer strictly necessary after the clone() fix in forward(),
#  but good practice to ensure clean data.)
val_dataset_clean = builder.build_dataset(
    configurations={'scalar': configurations[n_train:]},
    actions=actions[n_train:],
)

# Batch evaluation — much faster than single-graph loop
val_loader = PyGDataLoader(val_dataset_clean, batch_size=64, shuffle=False)

with torch.no_grad():
    for batch in val_loader:
        batch = batch.to(device)
        output = model(batch)
        all_pred.append(output['energy'].cpu())
        all_true.append(batch.y.cpu())

pred = torch.cat(all_pred)
true = torch.cat(all_true)

corr = energy_correlation(pred, true)
rel_err = relative_error(pred, true)

print(f'Validation Results:')
print(f'  Pearson correlation: {corr:.4f}')
print(f'  Mean relative error: {rel_err:.4f}')
print(f'  Pred range: [{pred.min():.1f}, {pred.max():.1f}]')
print(f'  True range: [{true.min():.1f}, {true.max():.1f}]')

fig, ax = plt.subplots(figsize=(7, 7))
ax.scatter(true.numpy(), pred.numpy(), alpha=0.3, s=10, color='#4488ff')
lims = [min(true.min(), pred.min()), max(true.max(), pred.max())]
ax.plot(lims, lims, 'r--', linewidth=2, label='Perfect prediction')
ax.set_xlabel(r'True $S_E[\phi]$')
ax.set_ylabel(r'Predicted $S_E[\phi]$')
ax.set_title(f'Energy Prediction (r = {corr:.4f})')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_aspect('equal')
plt.tight_layout()
plt.show()

## 6. Coupling Sweep for Critical Exponents

Scan m² at fixed λ to locate the phase transition and extract ν.
Run at multiple lattice sizes for finite-size scaling.

In [ ]:
# This cell is computationally expensive. Reduce n_configs or m2_steps for faster runs.
SWEEP_CONFIGS = 1000
M2_STEPS = 20
SWEEP_SIZES = [(8, 8), (16, 16)]  # Add (32, 32) for publication quality

# Extended range to capture the critical point (m²_c ≈ -2.1 from notebook 01)
m2_values = np.linspace(0.0, -2.5, M2_STEPS)
sweep_results = {}

for dims in SWEEP_SIZES:
    L = dims[0]
    print(f'\n=== Sweeping L={L} ===')
    lat = HypercubicLattice(LatticeConfig(dimensions=dims))
    obs = ObservableSet(lat)
    warm_phi = None  # warm-start between m² values

    mags, chis, xis_log, xis_sm = [], [], [], []

    for i, m2 in enumerate(m2_values):
        act = Phi4Action(lat, ScalarFieldConfig(mass_squared=m2, coupling=COUPLING))
        samp = MetropolisSampler(act, MCConfig(
            n_configs=SWEEP_CONFIGS, n_thermalization=1000,
            n_sweeps_between=10, seed=None))
        res = samp.generate(SWEEP_CONFIGS, initial_phi=warm_phi)
        warm_phi = res.configurations[-1].clone()

        n = len(res.configurations)
        M_samples = torch.tensor([res.configurations[j].mean().item() for j in range(n)])
        absM_samples = M_samples.abs()
        M2_samples = M_samples ** 2

        mag_mean, _ = jackknife_mean_error(absM_samples)
        V = lat.num_sites()
        chi = V * (M2_samples.mean().item() - absM_samples.mean().item()**2)

        # Average the connected correlator over configs, then extract ξ
        G_r_avg = torch.zeros(L // 2 + 1)
        n_corr = min(200, n)
        for j in range(n_corr):
            G_r_avg += obs.two_point_function(res.configurations[j], connected=True)
        G_r_avg /= n_corr

        # Log-slope estimator (old method, for comparison)
        xi_log = ObservableSet.correlation_length(G_r_avg)

        # Second-moment estimator (new, robust)
        xi_sm = ObservableSet.correlation_length_second_moment(G_r_avg, L)

        mags.append(mag_mean)
        chis.append(chi)
        xis_log.append(xi_log / L if xi_log < 1000 else 0.0)
        xis_sm.append(xi_sm / L)

        if (i+1) % 5 == 0:
            print(f'  m²={m2:.2f}: |M|={mag_mean:.3f}, χ={chi:.1f}, '
                  f'ξ_SM/L={xi_sm/L:.3f}, ξ_log/L={xi_log/L:.3f}, '
                  f'acc={res.acceptance_rate:.3f}')

    sweep_results[L] = {
        'mags': mags, 'chis': chis,
        'xi_over_L': xis_sm,       # Use second-moment as primary
        'xi_log_over_L': xis_log,  # Keep log-slope for comparison
    }

print('\nSweep complete!')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
colors = {'8': '#ff6644', '16': '#4488ff', '32': '#44bb88'}

for L, data in sweep_results.items():
    c = colors.get(str(L), '#ffffff')
    axes[0].plot(m2_values, data['mags'], 'o-', color=c, label=f'L={L}', markersize=4)
    axes[1].plot(m2_values, data['chis'], 's-', color=c, label=f'L={L}', markersize=4)
    axes[2].plot(m2_values, data['xi_over_L'], '^-', color=c, label=f'L={L}', markersize=4)

axes[0].set_xlabel(r'$m^2$')
axes[0].set_ylabel(r'$|\langle\phi\rangle|$')
axes[0].set_title('Order Parameter')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].set_xlabel(r'$m^2$')
axes[1].set_ylabel(r'$\chi$')
axes[1].set_title('Susceptibility')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

axes[2].set_xlabel(r'$m^2$')
axes[2].set_ylabel(r'$\xi / L$')
axes[2].set_title(r'$\xi/L$ Crossing (should cross at $m^2_c$)')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.suptitle(rf'Finite-Size Scaling: $\lambda={COUPLING}$', fontsize=14)
plt.tight_layout()
plt.show()

## 7. Save Results

Save the trained model and sweep data. When running on Colab with Google Drive mounted, everything is saved directly to your Drive — no download step needed.

In [ ]:
# Save model checkpoint
save_dir = Path('experiments/runs/colab_run')
save_dir.mkdir(parents=True, exist_ok=True)

torch.save({
    'model_state_dict': model.state_dict(),
    'history': history,
    'config': {
        'lattice_size': LATTICE_SIZE,
        'mass_squared': MASS_SQUARED,
        'coupling': COUPLING,
        'hidden_dim': HIDDEN_DIM,
        'n_mp_blocks': N_MP_BLOCKS,
    },
}, save_dir / 'model_final.pt')

# Save sweep results
import json
sweep_save = {}
for L, data in sweep_results.items():
    sweep_save[str(L)] = {
        'm2_values': m2_values.tolist(),
        'mags': data['mags'],
        'chis': data['chis'],
        'xi_over_L': data['xi_over_L'],
    }

with open(save_dir / 'sweep_results.json', 'w') as f:
    json.dump(sweep_save, f, indent=2)

print(f'Saved to {save_dir.resolve()}')
if IN_COLAB:
    print('Results are persisted on your Google Drive automatically.')